# 🎓 Modelo de Machine Learning — Previsão de Evasão (Graduação Híbrida)
### a instituicao | v3 — validação em origem deslizante

Prevê a probabilidade de um aluno da graduação híbrida **não se rematricular** no
trimestre seguinte, para que a equipe de retenção priorize quem abordar.

O modelo não existe para acertar uma métrica: ele existe para **ordenar uma fila**.
A equipe aborda ~200 alunos por ciclo, então a métrica que manda é **quantos evasores
reais caem no topo 200 da lista**.

**Configuração, e por que ela é essa** (medido em 17/09/2026, origem deslizante,
10 sementes, limiar travado em 0,30):

| Decisão | Motivo |
|---|---|
| Treino nas 3 coortes viáveis | +2,7 pp de `recall@200` contra treinar só na última |
| Features de AVA ranqueadas dentro da coorte | +2,4 pp — corrige escala que mudava a cada trimestre |
| Sem imputação de mediana | nulo em `AVA_*` é informação; o RF do sklearn ≥ 1.4 trata NaN nativamente |
| Split temporal (nunca aleatório) | é assim que o modelo é usado: treinado no passado, aplicado na turma nova |
| Peso da classe 2,0× | mantém `recall@0,30` acima do modelo antigo com metade dos alertas |
| Limiar fixo em 0,30 | decisão do projeto, não otimizado aqui |

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    recall_score,
    precision_score,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

LIMIAR = 0.30       # travado por decisao do projeto
FILA = 200          # capacidade da equipe de retencao por ciclo
SEMENTE = 42

## 1. Carregamento dos Dados

Base oficial única gerada por `querys/extrair_hibrido.py`, que roda
`querys/03_extracao_hibrido.sql` uma vez por origem e empilha o resultado.

Lemos o **CSV**, não o XLSX: o Excel já converteu código de `CURSO` em data e
corrompeu acentos em extrações passadas.

In [ ]:
# Aceita CSV ou XLSX. O CSV e o formato preferido (o Excel converte codigo de
# CURSO em data), mas o XLSX serve desde que o dtype seja forcado na leitura --
# sem isso CURSO volta como int64 e o modelo trata codigo de curso como escala.
DTYPE = {"ALUNO": str, "CURSO": str, "END_MUNICIPIO": str}

def carregar_base(nome="extracao_hibrido"):
    for pasta in ("../../tabelas", "tabelas"):
        csv, xlsx = f"{pasta}/{nome}.csv", f"{pasta}/{nome}.xlsx"
        if os.path.exists(csv):
            return csv, pd.read_csv(csv, encoding="utf-8-sig", dtype=DTYPE)
        if os.path.exists(xlsx):
            return xlsx, pd.read_excel(xlsx, dtype=DTYPE)
    raise FileNotFoundError(
        f"{nome}.csv/.xlsx nao encontrado em tabelas/. A base e gitignored "
        "(dado pessoal de aluno) -- rode querys/03_extracao_hibrido.sql uma vez "
        "por coorte e empilhe. Os parametros estao em querys/README.md."
    )

CSV, df = carregar_base()
print(f"Carregando: {CSV}")
print(f"Dimensões brutas: {df.shape[0]:,} linhas x {df.shape[1]} colunas\n")

print("Origens na base (observação -> alvo):")
print(df.groupby("COORTE").size().rename("linhas").to_string())
df.head(3)

## 2. Filtro Obrigatório de Elegibilidade (`ELEGIVEL == 1`)

Descarta quem concluiu ou foi transferido antes do corte — não há o que prever nesses
casos. Sem esse filtro o modelo aprende que quem ainda está no curso nunca evade.

In [ ]:
df_model = df[df["ELEGIVEL"] == 1].copy()

print(f"Elegíveis: {len(df_model):,}  |  Evasores: {int(df_model['EVADIU'].sum()):,}"
      f"  |  Taxa: {df_model['EVADIU'].mean():.2%}\n")

resumo = (
    df_model.groupby("COORTE")
    .agg(ALUNOS=("ALUNO", "count"), EVASORES=("EVADIU", "sum"))
    .assign(TAXA=lambda d: (100 * d.EVASORES / d.ALUNOS).round(2))
)
print(resumo.to_string())

## 3. Engenharia de Features de Risco Combinado

Duas etapas, **nesta ordem**:

1. **Sinais de risco a partir do valor cru** — `RISCO_SUMIU_AVA` testa "mais de 30 dias
   sem acessar", o que só faz sentido em dias de verdade.
2. **Ranqueamento do AVA dentro da coorte** — só depois.

O ranqueamento é a correção mais importante desta versão. `DT_CORTE` é o `DT_FIM`
formal do trimestre, e a distância entre o fim formal e o fim real da atividade muda a
cada período: a mediana de `AVA_DIAS_DESDE_ULT_ACESSO` é **0 dia em 2025/23, 15 em
2025/24 e 2 em 2026/21** (2025/24 fecha em 30/12, no recesso). A feature mais preditiva
do modelo estava numa escala diferente em cada turma, e o ROC-AUC desabava para 0,54
quando se treinava numa e testava em outra.

Trocar o valor bruto pelo **percentil dentro da própria coorte** resolve, e é legítimo
em produção: a fila é sempre pontuada de uma vez, então o rank está disponível no
momento da inferência.

In [ ]:
# ---- 1. sinais de risco, calculados sobre o valor CRU -----------------------
df_model["EST_CIVIL"] = (
    df_model["EST_CIVIL"].astype(str).str.upper()
    .replace({"NAN": "NAO_INFORMADO", "NONE": "NAO_INFORMADO"})
)

df_model["RISCO_NOTA_BAIXA"] = (df_model["NOTA_MEDIA"] < 5.0).astype(int)
df_model["RISCO_SUMIU_AVA"] = (df_model["AVA_DIAS_DESDE_ULT_ACESSO"] > 30).astype(int)
df_model["RISCO_COMBO_ACAD_AVA"] = df_model["RISCO_NOTA_BAIXA"] * df_model["RISCO_SUMIU_AVA"]
df_model["RATIO_GAP_ATIVOS"] = df_model["AVA_MAIOR_GAP_DIAS"] / (df_model["AVA_DIAS_ATIVOS"] + 1)
df_model["SESSOES_POR_DIA_ATIVO"] = df_model["AVA_QTD_SESSOES"] / (df_model["AVA_DIAS_ATIVOS"] + 1)
df_model["DESACELEROU_AVA"] = (df_model["AVA_TEND_SESSOES"] < 0.5).astype(int)

# ---- 2. agora sim, o nivel bruto do AVA vira percentil da coorte ------------
AVA_ESCALA_COORTE = [
    "AVA_DIAS_DESDE_ULT_ACESSO", "AVA_DIAS_ATE_1O_ACESSO", "AVA_MAIOR_GAP_DIAS",
    "AVA_QTD_SESSOES", "AVA_DIAS_ATIVOS", "AVA_TAXA_DIAS_ATIVOS",
    "AVA_SESSOES_RECENTES", "AVA_DIAS_ATIVOS_RECENTES", "AVA_TEND_SESSOES",
]

antes = df_model.groupby("COORTE")["AVA_DIAS_DESDE_ULT_ACESSO"].median()
for c in AVA_ESCALA_COORTE:
    if c in df_model.columns:
        df_model[c] = df_model.groupby("COORTE")[c].rank(pct=True)

print("Mediana de AVA_DIAS_DESDE_ULT_ACESSO ANTES do rank (em dias):")
print(antes.round(1).to_string())
print("\nDepois do rank, por construção, toda coorte tem mediana ~0,5 —")
print("é exatamente esse deslocamento entre turmas que o rank remove.\n")

# ---- 3. agrupamento de municipio (codigo IBGE nao e escala continua) --------
mun = df_model["END_MUNICIPIO"].value_counts()
principais_mun = mun[mun >= 15].index.tolist()
df_model["END_MUNICIPIO_AGRUPADO"] = df_model["END_MUNICIPIO"].apply(
    lambda x: str(x) if x in principais_mun else "OUTRO"
)
print(f"Municípios mantidos: {len(principais_mun)} (cauda longa agrupada em 'OUTRO')")

## 4. Separação de $X$ (Features) e $y$ (Target)

Fora de `X`: identificador, tudo que deriva do alvo e metadados de auditoria.

In [ ]:
colunas_remover = [
    "ALUNO",                    # identificador -- o modelo decora o aluno
    "CURSOU_ALVO",              # E o rotulo (vazamento puro)
    "SIT_ALVO",                 # deriva do alvo
    "SIT_FIM_OBS",              # situacao de controle
    "AUDITORIA_STATUS",         # auditoria de elegibilidade/desfecho
    "ELEGIVEL",                 # filtro de controle
    "EVADIU",                   # o rotulo
    "DT_INICIO_OBS", "DT_CORTE",  # metadados de auditoria
    "ANO_OBS", "PERIODO_OBS", "COORTE",  # identificam a origem, nao o aluno
    "NOME_CURSO",               # redundante com o codigo CURSO
    "NACIONALIDADE",            # ~100% 'BRASILEIRA', sem variancia
    "UNIDADE_FISICA",           # ~100% constante no hibrido
    "END_MUNICIPIO",            # substituido por END_MUNICIPIO_AGRUPADO
    "AVA_QTD_SESSOES", "AVA_DIAS_ATIVOS",          # redundantes com as derivadas
    "AVA_SESSOES_RECENTES", "AVA_DIAS_ATIVOS_RECENTES",

    # ==================================================================
    # CORTE DE RUIDO -- RODADA 1, medida em 17/09/2026
    # Origem deslizante, 8 sementes, teste na coorte 2026/21
    # (2.543 alunos, 173 evasores), limiar 0,30:
    #
    #                          colunas   recall@200   evasores   PR-AUC
    #   com os 4 blocos             74        53,8%       93,0    0,456
    #   sem os 4 blocos             64        54,9%       95,0    0,460
    # ==================================================================
    "TURNO",                    # 100% 'HIB' nesta base -- constante pura
    "MODALIDADE_CURSO",         # 89,1% 'Bach' -- variancia baixa demais
    "EST_CIVIL",                # 8 colunas apos encoding, sem sinal medido
    "SEXO",                     # fora do X por decisao etica do CLAUDE.md:
                                #   "usar como feature preditiva direciona
                                #   intervencao por genero/raca. Considere so
                                #   para auditoria de vies".
                                #   Continua em df_model e e usado no topico 9
                                #   para AUDITAR vies -- nunca para prever.

    # ==================================================================
    # CORTE DE RUIDO -- RODADA 2, medida em 19/09/2026
    #
    # Motivo: a matriz de correlacao do topico 14 mostrava uma linha em
    # branco e muita celula cinza. Investigando, 36 das 72 colunas tinham
    # importancia Gini abaixo de 0,2%, e 9 colunas CURSO_* tinham
    # importancia exatamente 0,0000. O peso morto era quase todo one-hot.
    #
    # Ablacao (8 sementes, mesma coorte de teste):
    #
    #                                  colunas   recall@200  evasores  PR-AUC
    #   modelo da rodada 1                  64       54,9%      95,0   0,4602
    #   sem DISC_REP_FREQ_ANT               63       54,5%      94,2   0,4557
    #   sem END_MUNICIPIO                   45       55,1%      95,4   0,4593
    #   sem CURSO                           45       55,4%      95,9   0,4623
    #   sem os tres (este corte)            27       55,5%      96,0   0,4678
    #   + sem PER_INGRESSO/ANOS_DE_CASA     25       54,4%      94,1   0,4670
    #
    # O ultimo cenario piorou -- PER_INGRESSO e ANOS_DE_CASA ficam.
    #
    # TROCA NO LIMIAR 0,30 (8 sementes, mesma coorte): o corte ganha na fila e
    # perde um pouco no limiar. Os dois numeros, porque a equipe vive os dois:
    #
    #                   recall@0,30      alertas   precisao   recall@200
    #   64 colunas    91,0% (158/173)       798      19,7%      54,9%
    #   27 colunas    90,0% (156/173)       738      21,1%      55,5%
    #
    # Sao 2 evasores a menos alcancados pelo limiar, contra 60 alarmes falsos a
    # menos e +1 evasor no topo 200. Como a equipe aborda ~200 por ciclo e nao
    # 738, quem decide na pratica e a fila -- por isso o corte fica.
    #
    # EFEITO COLATERAL BOM: tirar CURSO reduz o vies de genero do topico 9,
    # de 7,5 para 6,0 pontos de diferenca no recall entre F e M. O curso era
    # um dos caminhos do proxy (Engenharia de Controle tem 8,7% de mulheres,
    # Estetica tem 92,7%). Nao resolve sozinho: 6 pontos sobram.
    # ==================================================================
    "DISC_REP_FREQ_ANT",        # CONSTANTE: 1 unico valor (0.0) em 80,7% das
                                #   linhas, nulo nos outros 19,3%. Nenhum aluno
                                #   do hibrido tem reprovacao por frequencia.
                                #   E a linha em branco da matriz de correlacao
                                #   -- correlacao com coluna constante e
                                #   indefinida (divisao por desvio-padrao zero).
    "CURSO",                    # 35 colunas apos encoding, 9 delas com
                                #   importancia 0,0000. O sinal de curso ja
                                #   chega por IDADE_INGRESSO, HIST_* e CREDITOS.
    "END_MUNICIPIO_AGRUPADO",   # 19 colunas apos encoding, 1,3% de importancia
                                #   somada. Codigo de municipio nao informa
                                #   evasao depois que o AVA entra no modelo.
]

X = df_model.drop(columns=[c for c in colunas_remover if c in df_model.columns])
y = df_model["EVADIU"].astype(int)

print(f"Features em X: {X.shape[1]}\n")
print(X.columns.tolist())


## 5. Encoding de Categóricas — **sem imputação**

Aqui o notebook se afasta do processo padrão de propósito, seguindo o `CLAUDE.md`:

> Não impute média em `P{n}_*` nem em `DELTA_*`. Ali o nulo é informação: significa que
> o aluno não cursou nada naquele período — quase o próprio sinal de evasão.
> LightGBM, CatBoost e o RandomForest do sklearn ≥ 1.4 tratam NaN nativamente.

Imputar mediana num aluno sem registro de AVA o transforma num aluno médio, que é
apagar o sinal. O `RandomForestClassifier` recebe os `NaN` como estão.

In [ ]:
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
print(f"Categóricas ({len(cat_cols)}): {cat_cols}")

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

vazias = [c for c in X_encoded.columns if X_encoded[c].isna().all()]
if vazias:
    print(f"\nDescartando {len(vazias)} colunas 100% nulas: {vazias}")
    X_encoded = X_encoded.drop(columns=vazias)

import sklearn
print(f"\nscikit-learn {sklearn.__version__} (NaN nativo no RF exige >= 1.4)")
print(f"Dimensões finais de X: {X_encoded.shape[1]} colunas")

nulos = X_encoded.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print(f"\n{len(nulos)} colunas com nulo preservado (NÃO imputadas):")
print((100 * nulos / len(X_encoded)).round(1).head(12).to_string())

## 6. Divisão Temporal — treino no passado, teste na turma seguinte

**Split temporal, nunca aleatório.** Um split aleatório mistura turmas e infla a
métrica num problema que é temporal por natureza. O modelo será usado treinado no
passado e aplicado na coorte nova — a validação precisa imitar isso.

- **Treino:** 2025/23 + 2025/24
- **Teste:** 2026/21 (a última origem com alvo fechado)

2026/22 → 2026/23 não serve como origem: o trimestre alvo começou em 27/07/2026 e ainda
está em curso.

In [ ]:
COORTES_TREINO = ["2025/23", "2025/24"]
COORTE_TESTE = "2026/21"

mask_tr = df_model["COORTE"].isin(COORTES_TREINO).values
mask_te = (df_model["COORTE"] == COORTE_TESTE).values

X_train, X_test = X_encoded[mask_tr], X_encoded[mask_te]
y_train, y_test = y[mask_tr], y[mask_te]

# guardado para a suite de testes em lote sortear SO de quem o modelo nao viu
alunos_teste = df_model.loc[mask_te, "ALUNO"].tolist()

print(f"Treino ({' + '.join(COORTES_TREINO)}): {len(X_train):,} alunos | "
      f"Evasores: {int(y_train.sum()):,} ({y_train.mean():.2%})")
print(f"Teste  ({COORTE_TESTE}):            {len(X_test):,} alunos | "
      f"Evasores: {int(y_test.sum()):,} ({y_test.mean():.2%})")

repetidos = len(set(df_model.loc[mask_te, 'ALUNO']) & set(df_model.loc[mask_tr, 'ALUNO']))
print(f"\nAlunos do teste que também aparecem no treino: {repetidos:,} "
      f"({100*repetidos/mask_te.sum():.1f}%)")
print("Isso é esperado e não é vazamento: em produção o modelo reencontra o mesmo aluno")
print("em trimestres seguintes, com rótulo diferente. Auditado no tópico 9.")

## 7. Treinamento do Random Forest (Sensível a Custo)

Base desbalanceada (~7–12% de evasão), então `class_weight` proporcional, multiplicado
por **2,0×**. O multiplicador foi medido: mantém `recall@0,30` acima do modelo antigo
(93,1% contra 92,3%) com **quase metade dos alertas** (854 contra 1.351).

In [ ]:
PESO_EXTRA = 2.0

scale_pos = max(1.0, (len(y_train) - y_train.sum()) / max(1, y_train.sum())) * PESO_EXTRA

rf_model = RandomForestClassifier(
    n_estimators=600,
    max_depth=14,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight={0: 1.0, 1: float(scale_pos)},
    random_state=SEMENTE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)   # NaN vai direto, sem imputador

y_proba = rf_model.predict_proba(X_test)[:, 1]
print(f"✅ Treinado. Peso da classe evasão: {scale_pos:.2f}")

## 8. O Limiar de Decisão e o Tamanho da Fila

O limiar está **travado em 0,30** por decisão do projeto. A tabela abaixo existe para
mostrar o que cada corte custa, não para escolher outro.

A coluna decisiva é **`Alertas`**. Quando ela passa de 200, o limiar deixa de decidir
qualquer coisa na prática: a equipe vai abordar os 200 primeiros da fila de qualquer
jeito, e quem manda passa a ser a **ordem**, não o corte.

In [ ]:
def recall_em_k(y_true, proba, k):
    ordem = np.argsort(-proba)[:k]
    return y_true.values[ordem].sum() / max(1, y_true.sum())


linhas = []
for t in [0.50, 0.40, 0.35, 0.30, 0.25, 0.20]:
    yp = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, yp).ravel()
    linhas.append({
        "Limiar": f"{t:.2f}",
        "Recall": f"{recall_score(y_test, yp, zero_division=0):.1%}",
        "Precisão": f"{precision_score(y_test, yp, zero_division=0):.1%}",
        "F1": f"{f1_score(y_test, yp, zero_division=0):.1%}",
        "Alertas": int(yp.sum()),
        "Evasores achados": f"{tp} / {int(y_test.sum())}",
        "Falsos alarmes": int(fp),
    })

df_thresh = pd.DataFrame(linhas)
print(f"Base de teste: {len(y_test):,} alunos, {int(y_test.sum())} evasores")
print(f"Capacidade da equipe: ~{FILA} alunos por ciclo\n")
display(df_thresh)

print(f"\nIndependente do limiar, o topo {FILA} da fila captura "
      f"{recall_em_k(y_test, y_proba, FILA):.1%} dos evasores "
      f"({int(recall_em_k(y_test, y_proba, FILA) * y_test.sum())} alunos).")

## 9. Avaliação no Limiar 0,30

Três números, nesta ordem de importância:

1. **`recall@200`** — dos evasores reais, quantos caem no topo 200. É a capacidade real
   da equipe.
2. **`recall@10%`** — o mesmo, comparável entre bases de tamanhos diferentes.
3. **PR-AUC** — qualidade geral do ranking.

A acurácia não entra em nenhuma decisão: com ~7% de evasão, chutar "ninguém evade" dá
93% de acurácia e serve para nada.

In [ ]:
y_pred = (y_proba >= LIMIAR).astype(int)

print("=" * 70)
print(f"   RELATÓRIO DE CLASSIFICAÇÃO — limiar {LIMIAR} — coorte {COORTE_TESTE}")
print("=" * 70)
print(classification_report(y_test, y_pred,
                            target_names=["Permanência (0)", "Evasão (1)"],
                            zero_division=0))

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
r200 = recall_em_k(y_test, y_proba, FILA)
r10 = recall_em_k(y_test, y_proba, max(1, int(0.10 * len(y_test))))

print("=" * 70)
print(f"recall@{FILA}                    {r200:.2%}  "
      f"({int(r200 * y_test.sum())} dos {int(y_test.sum())} evasores)")
print(f"recall@10% da base           {r10:.2%}")
print(f"PR-AUC (Average Precision)   {pr_auc:.4f}")
print(f"ROC-AUC                      {roc_auc:.4f}")
print("=" * 70)

# =========================================================================
# AUDITORIA 1 -- MEMORIZACAO
# As coortes sao consecutivas, entao o mesmo aluno reaparece em trimestres
# diferentes com rotulo diferente. Isso reproduz producao, mas exige checar:
# se o modelo decorasse pessoas, iria muito melhor no grupo ja conhecido.
# =========================================================================
ja_visto = df_model.loc[mask_te, "ALUNO"].isin(
    set(df_model.loc[mask_tr, "ALUNO"])).values

print("\nAUDITORIA 1 — MEMORIZAÇÃO (desempenho por grupo)")
print("-" * 70)
for rotulo, m in [("Já visto no treino", ja_visto), ("Nunca visto", ~ja_visto)]:
    yt, pp = y_test.values[m], y_proba[m]
    print(f"{rotulo:20s} n={m.sum():5,}  evasão={yt.mean():6.2%}  "
          f"recall@{LIMIAR}={recall_score(yt, (pp >= LIMIAR).astype(int), zero_division=0):6.1%}  "
          f"PR-AUC={average_precision_score(yt, pp):.3f}")
print("-" * 70)
print("Se o modelo estivesse decorando, o grupo 'já visto' seria muito melhor.")

# =========================================================================
# AUDITORIA 2 -- VIES
# SEXO nao entra no X (ver topico 4). Mas tirar a coluna nao basta: CURSO e
# um informante forte de genero (Engenharia de Controle tem 8,7% de mulheres,
# Estetica tem 92,7%), entao a informacao entra pela porta dos fundos.
#
# Por isso a auditoria mede o desempenho POR GRUPO. Nao interessa se o modelo
# alerta mais um grupo que o outro -- interessa se ele ACERTA igual nos dois.
# Um grupo com recall menor recebe menos ligacoes da retencao sem que ninguem
# tenha decidido isso.
#
# Item aberto no CLAUDE.md: "Auditoria de vies (SEXO, UNIDADE_FISICA)".
# UNIDADE_FISICA nao entra aqui: e ~100% 'CAMPUS_01', nao ha grupo a comparar.
# =========================================================================
print("\nAUDITORIA 2 — VIÉS POR SEXO (a coluna está fora do X)")
print("-" * 70)
sexo_te = df_model.loc[mask_te, "SEXO"].fillna("NAO_INFORMADO").values
linhas_vies = []
for g in sorted(pd.unique(sexo_te)):
    m = sexo_te == g
    if m.sum() < 30:            # grupo pequeno demais para conclusao
        continue
    yt, pp = y_test.values[m], y_proba[m]
    pred_g = (pp >= LIMIAR).astype(int)
    topo = np.argsort(-y_proba)[:FILA]
    linhas_vies.append({
        "grupo": g,
        "alunos": int(m.sum()),
        "evasão real": f"{yt.mean():.2%}",
        "alertados": f"{pred_g.mean():.1%}",
        "recall": f"{recall_score(yt, pred_g, zero_division=0):.1%}",
        "precisão": f"{precision_score(yt, pred_g, zero_division=0):.1%}",
        f"no topo {FILA}": int(m[topo].sum()),
    })
display(pd.DataFrame(linhas_vies))

recalls = [float(l["recall"].rstrip("%")) for l in linhas_vies]
gap = max(recalls) - min(recalls)
print(f"Diferença de recall entre o melhor e o pior grupo: {gap:.1f} pontos.")
print("Acima de ~5 pontos, a fila está favorecendo um grupo — vale investigar")
print("qual feature está servindo de proxy antes de entregar a lista à retenção.")


## 10. Matrizes de Confusão Comparativas (0,50 vs 0,30)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, t, cmap, rot in [
    (axes[0], 0.50, "Blues", "Limiar padrão (0,50)"),
    (axes[1], LIMIAR, "Greens", f"Limiar do projeto ({LIMIAR:.2f})"),
]:
    yp = (y_proba >= t).astype(int)
    cm = confusion_matrix(y_test, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax, cbar=False,
                xticklabels=["Previsto: Fica", "Previsto: Evade"],
                yticklabels=["Real: Fica", "Real: Evade"])
    ax.set_title(f"{rot}\nRecall: {recall_score(y_test, yp, zero_division=0):.1%}  |  "
                 f"Alertas: {int(yp.sum()):,}", fontweight="bold")
    ax.set_ylabel("Classe real", fontweight="bold")
    ax.set_xlabel("Classe prevista", fontweight="bold")

plt.suptitle(f"Graduação Híbrida — coorte de teste {COORTE_TESTE} "
             f"({len(y_test):,} alunos, {int(y_test.sum())} evasores)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 11. Curva ROC e Curva Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, color="#1F4E78", lw=2.5, label=f"Random Forest (AUC = {roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1.5)
axes[0].set_title("Curva ROC", fontweight="bold")
axes[0].set_xlabel("Taxa de falsos positivos")
axes[0].set_ylabel("Recall (taxa de verdadeiros positivos)")
axes[0].legend(loc="lower right")

prec_c, rec_c, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(rec_c, prec_c, color="#009a73", lw=2.5, label=f"Random Forest (PR-AUC = {pr_auc:.3f})")
axes[1].axhline(y_test.mean(), color="crimson", linestyle="--", lw=1.5,
                label=f"Baseline: chutar tudo ({y_test.mean():.1%})")
axes[1].set_title("Curva Precision-Recall", fontweight="bold")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precisão")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

## 12. Importância das Features

O `CLAUDE.md` manda cortar o que não paga. As 6 colunas de frequência (`PERC_PRESENCA`
de T-1) foram medidas e **não pagaram** — ficaram dentro do intervalo de confiança nas
duas origens. Estão comentadas em `querys/03_extracao_hibrido.sql`, com o resultado da
medição registrado no cabeçalho do bloco.

In [ ]:
feat_df = (pd.DataFrame({"Feature": X_encoded.columns,
                         "Importancia": rf_model.feature_importances_})
           .sort_values("Importancia", ascending=False))

plt.figure(figsize=(10, 8))
sns.barplot(data=feat_df.head(20), x="Importancia", y="Feature",
            hue="Feature", palette="viridis", legend=False)
plt.title("Top 20 features — Graduação Híbrida", fontsize=13, fontweight="bold", pad=15)
plt.xlabel("Importância relativa (Gini)", fontsize=11, fontweight="bold")
plt.ylabel("")
plt.tight_layout()
plt.show()

acumulado = feat_df["Importancia"].cumsum()
n80 = int((acumulado <= 0.80).sum() + 1)
print(f"{n80} das {len(feat_df)} features concentram 80% da importância.")
print(f"\nAs {min(10, len(feat_df))} que menos pagam:")
print(feat_df.tail(10).to_string(index=False))

## 13. Salvamento do Artefato (`.pkl`)

Junto do modelo vai a lista de `ALUNO` da coorte de teste. Sem ela, os cadernos de
`area-testes/` sorteiam da base inteira e acabam medindo memória de treino em vez de
previsão — era o que acontecia na versão anterior, e inflava o recall reportado de 69%
para 95%.

In [ ]:
pkl_ml = "modelo_evasao_hibrido.pkl"
pkl_testes = os.path.join("..", "..", "area-testes", "hibrido", "modelo_evasao_hibrido.pkl")

artifact = {
    "model": rf_model,
    "imputer": None,                      # de proposito: NaN vai nativo
    "feature_names": X_encoded.columns.tolist(),
    "threshold": LIMIAR,
    "fila": FILA,
    "principais_municipios": principais_mun,
    "colunas_remover": colunas_remover,
    "ava_escala_coorte": AVA_ESCALA_COORTE,
    "coortes_treino": COORTES_TREINO,
    "coorte_teste": COORTE_TESTE,
    "alunos_teste": alunos_teste,         # para a suite sortear so de quem ele nao viu
    "metadata": {
        "modalidade": "Graduação Híbrida",
        "validacao": "split temporal (origem deslizante)",
        "n_features": int(X_encoded.shape[1]),
        "n_treino": int(len(X_train)),
        "n_teste": int(len(X_test)),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "recall_no_limiar": float(recall_score(y_test, y_pred, zero_division=0)),
        "precisao_no_limiar": float(precision_score(y_test, y_pred, zero_division=0)),
        "alertas_no_limiar": int(y_pred.sum()),
        "recall_at_200": float(r200),
        "recall_at_10pct": float(r10),
        "vies_gap_recall_sexo": float(gap),   # topico 9, auditoria 2
    },
}

# compress=3 porque sao 600 arvores de profundidade 14: sem compressao o
# arquivo passa de 21 MB, e sao DUAS copias versionadas (aqui e em
# area-testes/), o que colocaria 43 MB no git a cada retreino.
joblib.dump(artifact, pkl_ml, compress=3)
print(f"🎉 Salvo em: {pkl_ml}  ({os.path.getsize(pkl_ml)/(1024*1024):.1f} MB)")

if os.path.isdir(os.path.dirname(pkl_testes)):
    joblib.dump(artifact, pkl_testes, compress=3)
    print(f"🎉 Copiado para: {pkl_testes}")
else:
    print(f"⚠️  AVISO: {os.path.dirname(pkl_testes)} nao existe -- copia NAO feita.")

print("\nMétricas gravadas no artefato:")
for k, v in artifact["metadata"].items():
    print(f"  {k:24s} {v}")


## 14. Matriz de Correlação de Pearson

Três entregas: o heatmap triangular, o ranking de correlação com `EVADIU`
(`+r` = risco, `−r` = retenção) e o diagnóstico de multicolinearidade (`|r| ≥ 0,70`).

O PNG oficial vai para `relatorios/matriz-correlacao/matriz_correlacao_hibrido.png`.

In [ ]:
num_cols = [c for c in X.select_dtypes(include=[np.number]).columns if c != "CURSO"]
df_corr = df_model[num_cols].copy()
df_corr.insert(0, "EVADIU", y)
corr = df_corr.corr()

plt.figure(figsize=(11, 9))  # 11x9in @150dpi = 1650x1350 px
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", vmax=0.4, vmin=-0.4, center=0,
            square=True, linewidths=0.5, annot=True, fmt=".2f",
            annot_kws={"size": 6}, cbar_kws={"shrink": 0.8})
plt.title(f"Matriz de Correlação de Pearson — Graduação Híbrida\n"
          f"({len(df_model):,} alunos elegíveis, 3 coortes)",
          fontsize=12, fontweight="bold", pad=14)
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()

destino = os.path.join("..", "..", "relatorios", "matriz-correlacao")
if os.path.isdir(destino):
    saida = os.path.join(destino, "matriz_correlacao_hibrido.png")
    plt.savefig(saida, dpi=150)   # sem bbox_inches: ele recorta e quebra o padrao
    print(f"PNG oficial salvo em: {saida}")
plt.show()

In [ ]:
# --- ranking de correlacao com o alvo ---------------------------------------
corr_alvo = X_encoded.corrwith(y).dropna().sort_values()
ranking = pd.concat([corr_alvo.head(8), corr_alvo.tail(8)])
cores = ["#2B6CB0" if v < 0 else "#C53030" for v in ranking.values]

plt.figure(figsize=(12, 6))
barras = plt.barh(ranking.index, ranking.values, color=cores,
                  edgecolor="black", linewidth=0.5)
plt.axvline(0, color="gray", linestyle="--", linewidth=0.8)
plt.title("Correlação linear com a evasão — Graduação Híbrida",
          fontsize=12, fontweight="bold", pad=15)
plt.xlabel("Coeficiente de Pearson (r)   —   azul retém, vermelho arrisca",
           fontsize=10, fontweight="bold")
for b in barras:
    w = b.get_width()
    plt.text(w + (0.005 if w >= 0 else -0.005), b.get_y() + b.get_height() / 2,
             f"{w:+.2f}", va="center", ha="left" if w >= 0 else "right",
             fontsize=9, fontweight="bold", color="#2D3748")
plt.xlim(ranking.min() - 0.06, ranking.max() + 0.06)
plt.tight_layout()
plt.show()

# --- multicolinearidade ------------------------------------------------------
feat = [c for c in df_corr.columns if c != "EVADIU"]
pares = [
    {"Feature 1": a, "Feature 2": b, "r": round(corr.loc[a, b], 4)}
    for i, a in enumerate(feat) for b in feat[i + 1:]
    if abs(corr.loc[a, b]) >= 0.70
]
df_colin = (pd.DataFrame(pares).reindex(
    pd.DataFrame(pares)["r"].abs().sort_values(ascending=False).index)
    if pares else pd.DataFrame(columns=["Feature 1", "Feature 2", "r"]))

print("=" * 70)
print("DIAGNÓSTICO DE MULTICOLINEARIDADE (|r| >= 0,70)")
print("=" * 70)
print(df_colin.to_string(index=False) if not df_colin.empty
      else "Nenhum par com multicolinearidade severa.")